In [7]:
import pandas as pd
import os   
import re

In [8]:
def asegurar_columnas_base(df, columnas_base):
    """
    Asegura que el DF tenga las columnas con los nombres en `columnas_base`.
    Si no existen, intenta renombrar <columna>UM -> <columna>.
    Lanza KeyError si aún faltan columnas después del intento.
    """
    # 1) ¿Faltan columnas base?
    faltantes = [c for c in columnas_base if c not in df.columns]

    if faltantes:
        # 2) Intentar renombrar las que existan como <base>UM -> <base>
        ren = {f"{c}UM": c for c in faltantes if f"{c}UM" in df.columns}
        if ren:
            df.rename(columns=ren, inplace=True)

        # 3) Verificar nuevamente
        faltantes = [c for c in columnas_base if c not in df.columns]
        if faltantes:
            raise KeyError(f"Siguen faltando columnas: {faltantes}")

    return df

def Read_data(file_path, filepath2):
    columnas_a_convertir = ['Alcohol','Otras Sustancias','Tabaco','Marihuana','Cocaína','Inhalables','Metanfetaminas','Alucinógenos','Medicamentos','Opioides']
    df = pd.read_csv(file_path)
    df = df[df['Año'] > 2013]
    df2 = pd.read_csv(filepath2)
    df2.drop(columns=['DrogaImpacto'], inplace=True)
    df2 = df2.rename(columns={"Sexo": "SexoId"})
    df2 = asegurar_columnas_base(df2, columnas_a_convertir)
    df2[columnas_a_convertir] = df2[columnas_a_convertir].replace({1: True, 0: False})
    return df, df2

def concat_data(df1, df2):
    df = pd.concat([df1, df2],axis = 0, ignore_index=True)
    df["Edad_Años"] = df["Edad_Años"].combine_first(df["Edad_años"])
    df = df.drop(columns=["Edad_años"])
    df["EdadInicioCocaína"] = df["EdadInicioCocaína"].combine_first(df["EdadInicioCocaina"])
    df = df.drop(columns=["EdadInicioCocaina"])
    return df

def safe_convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return None

def Grupos(df):
    cols_derecha_EdadInicio = [
        # Marihuana
        'EdadInicioMarihuana', 'EdadInicioHachis',

        # Cocaína
        'EdadInicioCocaína', 'EdadInicioCrack',
        'EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)',

        # Inhalables
        'EdadInicioSolventes y removedores', 'EdadInicioPegamento',
        'EdadInicioEsmaltes y pinturas',
        'EdadInicioOtros (aire comprimido, gasolinas y combustibles)',

        # Metanfetaminas
        'EdadInicioAnfetaminas', 'EdadInicioMetanfetaminas',
        'EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        'EdadInicioOtros (derivados anfetaminicos)',

        # Alucinógenos
        'EdadInicioLSD', 'EdadInicioPlantas alucinogenas y derivados',
        'EdadInicioOtras (PCP, ketamina, excepto metanfetamina)',

        # Medicamentos
        'EdadInicioBenzodiazepinas', 'EdadInicioRohypnol',
        'EdadInicioOtras (sedantes hiptnoticos, GHB)',
        'EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)',

        # Opioides
        'EdadInicioHeroina',
        'EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)',
        'EdadInicioOpio y opiodes (morfina, codeina)'
    ]

    for col in cols_derecha_EdadInicio:
        df[col] = df[col].apply(safe_convert_to_float)
    # Agrupación por edad de inicio
    df['EdadInicioMarihuana'] = df[['EdadInicioMarihuana', 'EdadInicioHachis']].min(axis=1)
    df['EdadInicioCocaína'] = df[['EdadInicioCocaína', 'EdadInicioCrack', 'EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)']].min(axis=1)
    df['EdadInicioInhalables'] = df[['EdadInicioSolventes y removedores', 'EdadInicioPegamento', 'EdadInicioEsmaltes y pinturas', 'EdadInicioOtros (aire comprimido, gasolinas y combustibles)']].min(axis=1)
    df['EdadInicioMetanfetaminas'] = df[['EdadInicioAnfetaminas', 'EdadInicioMetanfetaminas', 'EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'EdadInicioOtros (derivados anfetaminicos)']].min(axis=1)
    df['EdadInicioAlucinógenos'] = df[['EdadInicioLSD', 'EdadInicioPlantas alucinogenas y derivados', 'EdadInicioOtras (PCP, ketamina, excepto metanfetamina)']].min(axis=1)
    df['EdadInicioMedicamentos'] = df[['EdadInicioBenzodiazepinas', 'EdadInicioRohypnol', 'EdadInicioOtras (sedantes hiptnoticos, GHB)', 'EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)']].min(axis=1)
    df['EdadInicioOpioides'] = df[['EdadInicioHeroina', 'EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)', 'EdadInicioOpio y opiodes (morfina, codeina)']].min(axis=1)


    cols_edad_no_finales = [
        'EdadInicioHachis',
        'EdadInicioCrack',
        'EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)',
        'EdadInicioSolventes y removedores',
        'EdadInicioPegamento',
        'EdadInicioEsmaltes y pinturas',
        'EdadInicioOtros (aire comprimido, gasolinas y combustibles)',
        'EdadInicioAnfetaminas',
        'EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)',
        'EdadInicioOtros (derivados anfetaminicos)',
        'EdadInicioLSD',
        'EdadInicioPlantas alucinogenas y derivados',
        'EdadInicioOtras (PCP, ketamina, excepto metanfetamina)',
        'EdadInicioBenzodiazepinas',
        'EdadInicioRohypnol',
        'EdadInicioOtras (sedantes hiptnoticos, GHB)',
        'EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)',
        'EdadInicioHeroina',
        'EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)',
        'EdadInicioOpio y opiodes (morfina, codeina)'
    ]


    df.drop(columns=cols_edad_no_finales, inplace=True)
    return df

def main():
    list_files_2014 = ['\\data\\ECERIECS_AlgunaVezGrupos.csv', '\\data\\ECERIECS(SI)_AlgunaVezGrupos.csv', '\\data\\ECERIECS_UltimoMesGrupos.csv', '\\data\\ECERIECS(SI)_UltimoMesGrupos.csv', '\\data\\ECERIECS_DrogaImpactoGrupos.csv', '\\data\\ECERIECS(SI)_DrogaImpactoGrupos.csv']
    list_files_2004 = ['\\data\\EI_AlgunaVez_2004_2013(GrupoLegalesIlegales).csv','\\data\\EI_AlgunaVez_2004_2013(GrupoIlegales).csv', '\\data\\EI_UltimoMes_2004_2013(GrupoLegalesIlegales).csv', '\\data\\EI_UltimoMes_2004_2013(GrupoIlegales).csv', '\\data\\EI_DROGAIMP_2004_2013(GrupoLegalesIlegales).csv', '\\data\\EI_DROGAIMP_2004_2013(GrupoIlegales).csv']
    for (col,col2) in zip(list_files_2014, list_files_2004):
        df1, df2 = Read_data(f'{os.getcwd()}{col}', f'{os.getcwd()}{col2}')
        df = concat_data(df1, df2)
        nombre = re.search(r'[^\\]+$', col).group()
        print(nombre)
        df = Grupos(df)
        df = df.rename(columns={'EdadInicioAlcohol': 'EdadInicio_Alcohol', 'EdadInicioTabaco': 'EdadInicio_Tabaco', 'EdadInicioMarihuana': 'EdadInicio_Marihuana', 'EdadInicioCocaína': 'EdadInicio_Cocaína', 'EdadInicioMetanfetaminas': 'EdadInicio_Metanfetaminas', 'EdadInicioOpioides': 'EdadInicio_Opioides', 'EdadInicioInhalables': 'EdadInicio_Inhalables', 'EdadInicioAlucinógenos': 'EdadInicio_Alucinógenos', 'EdadInicioMedicamentos': 'EdadInicio_Medicamentos', 'EdadInicioOtras Sustancias': 'EdadInicio_Otras Sustancias'})
        df.to_csv(f'{os.getcwd()}\\result\\{nombre}', index=False)
    return df

In [9]:
df = main()

C:\Users\franc\AppData\Local\Temp\ipykernel_29096\59788959.py:27: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(filepath2)


ECERIECS_AlgunaVezGrupos.csv


C:\Users\franc\AppData\Local\Temp\ipykernel_29096\59788959.py:27: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(filepath2)


ECERIECS(SI)_AlgunaVezGrupos.csv


C:\Users\franc\AppData\Local\Temp\ipykernel_29096\59788959.py:27: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(filepath2)


...

ECERIECS(SI)_UltimoMesGrupos.csv


C:\Users\franc\AppData\Local\Temp\ipykernel_29096\59788959.py:27: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(filepath2)


ECERIECS_DrogaImpactoGrupos.csv


C:\Users\franc\AppData\Local\Temp\ipykernel_29096\59788959.py:27: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(filepath2)


ECERIECS(SI)_DrogaImpactoGrupos.csv
